In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from google import genai
client = genai.Client()

In [5]:
import os
from openai import OpenAI

# Configuración para usar Gemini a través de la API de compatibilidad de OpenAI
openai_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [6]:
response = openai_client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[{"role": "user", "content": "di 'funciona' si me recibes"}]
)

print(response.choices[0].message.content)

funciona


In [7]:
def llm(prompt):
    response = openai_client.chat.completions.create(
        model="gemini-2.5-flash",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [16]:
llm('hey whats up')

"Hey! Not much, just hanging out in the digital world and ready to help you out. How's it going with you? What's on your mind today?"

In [8]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

That's great you're interested in joining! To give you the most accurate information, I'll need a little more detail about the specific course you're referring to.

Could you please tell me:

1.  **The name of the course?**
2.  **Where you discovered it** (e.g., a specific website, platform, university, or organization)?

Once I have that, I can help you check the enrollment status, application deadlines, and whether it has rolling admissions or a fixed start date!


In [9]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [10]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [11]:
question = "I just discovered the course. Can I join now?"
answer = llm(prompt)
print(answer)

Yes, but if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [12]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [3]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [25]:
courses_raw

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 404},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 85},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

In [4]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1350

In [26]:
documents[1100]

{'id': 'ed90e0f589',
 'course': 'machine-learning-zoomcamp',
 'section': 'Module 5. Deploying Machine Learning Models',
 'question': 'Bind for 0.0.0.0:9696 failed: port is already allocated',
 'answer': 'I was getting the following error when I rebuilt the Docker image, although the port was not allocated, and it was working fine.\n\nError message:\n\n```\nError response from daemon: driver failed programming external connectivity on endpoint beautiful_tharp (875be95c7027cebb853a62fc4463d46e23df99e0175be73641269c3d180f7796): Bind for 0.0.0.0:9696 failed: port is already allocated.\n```\n\n\n\nThe issue can be resolved by running the following command:\n\n```bash\ndocker kill $(docker ps -q)\n```\n\nFor more information, refer to the [GitHub issue on Docker for Windows](https://github.com/docker/for-win/issues/2722).'}

In [1]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

NameError: name 'documents' is not defined

In [28]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [29]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [30]:
search_results=search(question)

In [31]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [55]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [32]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [33]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [35]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [36]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [52]:
response = openai_client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)

Yes, you can join and start learning now.

You don't need to register or receive a confirmation email; you're accepted. You can simply start learning and submitting homework (while the submission forms are open).

However, please be aware of the following regarding certificates:

*   **Certificate Eligibility:** If you want to receive a certificate, you must submit your project while submissions are still being accepted. Certificates are only awarded if you finish the course with a "live" cohort, as it involves peer-reviewing three capstones after submitting your project, which can only be done while the course is running.
*   **Self-Paced Mode:** We do not award certificates for following the course in a self-paced mode outside of the active cohort timeline.

You can start whenever you want. The videos and GitHub materials are available. You should begin by consulting the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](http

In [46]:
response.choices[0].message.content

'Yes, you can join and start learning now. All course materials (videos, notebooks, and the GitHub repository) are available for you to follow at your own pace whenever you want.\n\nHowever, if you want to **receive a certificate**, you must:\n*   Finish the course with a "live" cohort.\n*   Submit your project while submissions are still being accepted.\n*   Be able to peer-review other projects, which is only possible when the course is actively running (after the submission form closes and the peer-review list is compiled).\n\nIf the current live cohort\'s deadlines for project submission and peer-review have passed, you won\'t be eligible for a certificate by joining now in a self-paced mode. You would need to wait for the next live cohort, which is scheduled for **Summer 2027**.\n\nYou can check the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/) for current deadlines.'

In [47]:
response.usage

CompletionUsage(completion_tokens=211, prompt_tokens=514, total_tokens=1733, completion_tokens_details=None, prompt_tokens_details=None)

In [53]:
input_price = 0.3 / 1_000_000
output_price = 2.50 / 1_000_000

cost = (
    response.usage.prompt_tokens * input_price +
    response.usage.completion_tokens * output_price
)

cost

0.0008792

In [ ]:
response = openai_client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)

In [57]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=message_history
)

In [58]:
print(response.choices[0].message.content)

Yes, you can join now.

However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. You can start whenever you want as the videos and GitHub materials are available.


In [62]:
def llm(instructions, user_prompt, model="gemini-2.5-flash"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=message_history
)

    return response.choices[0].message.content

In [63]:
def rag(query, model="gemini-2.5-flash"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [64]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can join now. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. You can start whenever you want, as the videos and GitHub materials are available.
